# VA-Driven Behaviors: Colab Diagnostic Tool

This notebook allows you to run the **Valence-Arousal Behavioral Pipeline** with GPU acceleration. It is designed to verify the Professor's "Change-Ratio" (Momentum) algorithm using the new windowed offline mode.

### 1. Prerequisites
Ensure your Runtime type is set to **GPU**:
`Runtime` -> `Change runtime type` -> `Hardware accelerator` -> `T4 GPU (or better)`

In [ ]:
# @title 2. Setup Environment & Audio Decoders
import os
from google.colab import files

print("Installing System Audio Decoders (FFmpeg)...")
!apt-get install -y ffmpeg

print("Installing Python dependencies...")
!pip install librosa opencv-python numpy scipy torch torchvision

# Create project structure if it doesn't exist
os.makedirs('models', exist_ok=True)
os.makedirs('samples', exist_ok=True)
os.makedirs('config', exist_ok=True)

print("Setup Complete!")

### 3. Calibration (Run this once)
To remove the "fallback threshold" warning and get accurate results, run this cell to calibrate the model to your specific test videos.

In [ ]:
# @title 3. Generate Calibration File
# This scans your samples folder to set the correct emotional baselines
!python pipeline/generate_calibration.py samples/ --output config/calibration.json --device cuda

In [ ]:
# @title 4. Run Momentum Analysis (GPU Accelerated)
# Change 'video67.mp4' to your actual video filename in samples/
VIDEO_NAME = "video67.mp4"

# NOTE: We removed --sparse because the GPU can handle full-frame analysis for better accuracy!
!python run_offline.py samples/{VIDEO_NAME} --windowed --device cuda

### ⚠️ Note for Teammate: Audio Extraction

We are currently observing a behavior where `librosa` fails to load audio directly from the `.mp4` container on this environment, even with FFmpeg installed. 

**Current Handling**: The pipeline is modality-robust—it will use zeros for audio and proceed with the visual-only analysis. This is sufficient to verify the momentum triggers and behavior mappings.

**Future Fix**: If high-fidelity audio triggers are required for the final paper, please investigate the `audioread`/`ffmpeg` interface or use a preprocessing step to convert the video to a `.wav` mono file before loading.